# Dev Baseline Analysis (Pipeline v1)

This notebook analyzes the full dev-split baseline run of the combined mapper + verifier pipeline.

**Input:** `results/dev_baseline_meaningful.csv` and `results/dev_baseline_all.csv`

**Source script:** `src/pipeline/run_dev_baseline.py` (all dev sentences from `hi_hdtb-ud-dev.conllu`)

Analysis only. No mapper, verifier, or rule changes.

**Decision labels:**
- `confirmed` / `ambiguous`: verifier-backed
- `mapping_hypothesis`: unverified mapper guess (UD label only)
- `no_decision`: no usable Karaka candidate

## 1. Load CSV Files

In [ ]:
import csv
from collections import Counter
from pathlib import Path

MEANINGFUL_PATH = Path("../results/dev_baseline_meaningful.csv")
ALL_PATH = Path("../results/dev_baseline_all.csv")


def load_csv(filepath):
    with open(filepath, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


meaningful = load_csv(MEANINGFUL_PATH)
all_rows = load_csv(ALL_PATH)

total_tokens = len(all_rows)
print(f"Total tokens (all):        {total_tokens}")
print(f"Meaningful rows:           {len(meaningful)}")
print(f"Sentences (unique sent_id): {len(set(r['sent_id'] for r in all_rows))}")

## 2. Summary Counts

In [ ]:
final_counts_all = Counter(row["final_decision"] for row in all_rows)
final_counts_meaningful = Counter(row["final_decision"] for row in meaningful)
mapper_status_counts = Counter(row["mapper_status"] for row in all_rows)
rule_counts = Counter(row["verifier_rule_id"] for row in all_rows if row["verifier_rule_id"])
deprel_counts = Counter(row["deprel"] for row in all_rows)

print("final_decision (all tokens):")
for decision, count in sorted(final_counts_all.items()):
    pct = 100 * count / total_tokens
    print(f"  {decision:<22} {count:>6}  ({pct:.2f}%)")
print()

print("final_decision (meaningful only):")
for decision, count in sorted(final_counts_meaningful.items()):
    pct = 100 * count / len(meaningful)
    print(f"  {decision:<22} {count:>6}  ({pct:.2f}%)")
print()

print("mapper_status (all tokens):")
for status, count in sorted(mapper_status_counts.items()):
    pct = 100 * count / total_tokens
    print(f"  {status:<22} {count:>6}  ({pct:.2f}%)")
print()

print("verifier_rule_id (tokens where a rule fired):")
rule_total = sum(rule_counts.values())
for rule_id, count in sorted(rule_counts.items()):
    pct_all = 100 * count / total_tokens
    pct_rules = 100 * count / rule_total
    print(f"  {rule_id:<6} {count:>6}  ({pct_all:.2f}% of all, {pct_rules:.1f}% of rule hits)")
print()

print("Top deprels (all tokens):")
for deprel, count in deprel_counts.most_common(15):
    pct = 100 * count / total_tokens
    print(f"  {deprel:<14} {count:>6}  ({pct:.2f}%)")

## 3. Example Rows by final_decision

In [ ]:
DISPLAY_COLS = [
    "sent_id",
    "token_form",
    "deprel",
    "case_marker",
    "final_decision",
    "final_candidates",
    "verifier_rule_id",
    "mapper_status",
]


def show_examples(example_rows, title, max_examples=5):
    print(title)
    print("=" * 70)
    if not example_rows:
        print("(no rows)")
        print()
        return
    for i, row in enumerate(example_rows[:max_examples], start=1):
        print(f"Example {i} ({row['sent_id']}, {row['token_form']})")
        print(f"  Sentence: {row['sentence_text']}")
        for col in DISPLAY_COLS:
            print(f"  {col}: {row[col]}")
        print()


confirmed_rows = [r for r in meaningful if r["final_decision"] == "confirmed"]
ambiguous_rows = [r for r in meaningful if r["final_decision"] == "ambiguous"]
hypothesis_rows = [r for r in meaningful if r["final_decision"] == "mapping_hypothesis"]

show_examples(confirmed_rows, f"Confirmed ({len(confirmed_rows)} total)")
show_examples(ambiguous_rows, f"Ambiguous ({len(ambiguous_rows)} total)")
show_examples(hypothesis_rows, f"Mapping hypothesis ({len(hypothesis_rows)} total)")

## 4. Summary

Full baseline report: `docs/verifier_v1_dev_baseline.md`

Regenerate dev CSVs:

```bash
python src/pipeline/run_dev_baseline.py
```